1. Import + load

In [4]:
import pandas as pd
import gc

df = pd.read_csv('../ex04/fines.csv')

2. Iterations (5 xil usul)

2.1 for + iloc (eng sekin

In [5]:
%%timeit

def loop_iloc(df):
    result = []
    for i in range(len(df)):
        row = df.iloc[i]
        if row['Refund'] == 0:
            result.append(0)
        else:
            result.append(row['Fines'] / row['Refund'] * row['Year'])
    return result

df['calc_iloc'] = loop_iloc(df)

31.8 ms ± 1.97 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


2.2 iterrows()

In [6]:
%%timeit

def loop_iterrows(df):
    result = []
    for _, row in df.iterrows():
        if row['Refund'] == 0:
            result.append(0)
        else:
            result.append(row['Fines'] / row['Refund'] * row['Year'])
    return result

df['calc_iterrows'] = loop_iterrows(df)

19.2 ms ± 891 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


2.3 apply + lambda

In [7]:
%%timeit

df['calc_apply'] = df.apply(
    lambda row: 0 if row['Refund'] == 0 else row['Fines'] / row['Refund'] * row['Year'],
    axis=1
)

6.06 ms ± 941 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


2.4 vectorized (Series) => bu usulda tez


In [8]:
%%timeit

df['calc_vector'] = (df['Fines'] / df['Refund'].replace(0, 1)) * df['Year']

159 μs ± 7.27 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


2.5 values (numpy style) => Bunisi eng tezi
    

In [9]:
%%timeit

df['calc_values'] = (df['Fines'].values / df['Refund'].replace(0, 1).values) * df['Year'].values

99.4 μs ± 5.78 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


3. Indexing

3.1 indexsiz

In [10]:
%%timeit
df[df['CarNumber'] == 'O136HO197RUS']

274 μs ± 19.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


3.2 index qo‘yish

In [11]:
df_indexed = df.set_index('CarNumber')

3.3 indexed search

In [12]:
%%timeit
df_indexed.loc['O136HO197RUS']

40 μs ± 3.35 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


4. Downcasting

4.1 original

In [13]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CarNumber      930 non-null    str    
 1   Refund         930 non-null    int64  
 2   Fines          930 non-null    float64
 3   Make           930 non-null    str    
 4   Model          919 non-null    str    
 5   Year           930 non-null    int64  
 6   calc_iloc      930 non-null    float64
 7   calc_iterrows  930 non-null    float64
 8   calc_apply     930 non-null    float64
 9   calc_vector    930 non-null    float64
 10  calc_values    930 non-null    float64
dtypes: float64(6), int64(2), str(3)
memory usage: 211.2 KB


4.2 copy

In [14]:
optimized_df = df.copy()

4.3 float → float32

In [15]:
for col in optimized_df.select_dtypes(include='float64').columns:
    optimized_df[col] = pd.to_numeric(optimized_df[col], downcast='float')

4.4 int → minimal

In [16]:
for col in optimized_df.select_dtypes(include='int64').columns:
    optimized_df[col] = pd.to_numeric(optimized_df[col], downcast='integer')

4.5 natija

In [17]:
optimized_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CarNumber      930 non-null    str    
 1   Refund         930 non-null    int8   
 2   Fines          930 non-null    float32
 3   Make           930 non-null    str    
 4   Model          919 non-null    str    
 5   Year           930 non-null    int16  
 6   calc_iloc      930 non-null    float64
 7   calc_iterrows  930 non-null    float64
 8   calc_apply     930 non-null    float64
 9   calc_vector    930 non-null    float64
 10  calc_values    930 non-null    float64
dtypes: float32(1), float64(5), int16(1), int8(1), str(3)
memory usage: 195.7 KB


5. Categories

In [18]:
for col in optimized_df.select_dtypes(include='object').columns:
    optimized_df[col] = optimized_df[col].astype('category')

optimized_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   CarNumber      930 non-null    category
 1   Refund         930 non-null    int8    
 2   Fines          930 non-null    float32 
 3   Make           930 non-null    category
 4   Model          919 non-null    category
 5   Year           930 non-null    int16   
 6   calc_iloc      930 non-null    float64 
 7   calc_iterrows  930 non-null    float64 
 8   calc_apply     930 non-null    float64 
 9   calc_vector    930 non-null    float64 
 10  calc_values    930 non-null    float64 
dtypes: category(3), float32(1), float64(5), int16(1), int8(1)
memory usage: 79.4 KB


/tmp/ipykernel_12980/3662594320.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in optimized_df.select_dtypes(include='object').columns:


6. Memory clean

In [19]:
%reset_selective df